### Aufgabe 3.1

In [141]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

from scipy.optimize import curve_fit
from uncertainties import ufloat

In [142]:
df = pd.read_excel('data/data.xlsx')

df.set_index("Ordnung", inplace=True)
df = df[df.columns[:3]]

sigma_s = 0.05  # uncertainty of measured values
sigma_l = 0.1   # uncertainty of distances

for col in df.columns:
    df[str(col) + "_u"] = [ufloat(v, sigma_s) for v in df[col]]

for col in df.iloc[:,3:]:
    l = ufloat(col.split("_")[0], sigma_l)
    df[str(col) + "_calc"] = df[col] / l

df = df[df.index > 0]
df.sort_values(by="Ordnung", inplace=True)
df = df[["80_u_calc", "120_u_calc", "160_u_calc"]]
df.head()

,80_u_calc,120_u_calc,160_u_calc
Ordnung,,,
1,0.0037+/-0.0006,0.0033+/-0.0004,0.00375+/-0.00031
2,0.0081+/-0.0006,0.0083+/-0.0004,0.00875+/-0.00031
3,0.0125+/-0.0006,0.0117+/-0.0004,0.01313+/-0.00031
4,0.0163+/-0.0006,0.0158+/-0.0004,0.01688+/-0.00031
5,0.0212+/-0.0006,0.0200+/-0.0004,0.02125+/-0.00031


In [143]:
def linear(x, m, c):
    return m*x + c

df["avg"] = (df["80_u_calc"] + df["120_u_calc"] + df["160_u_calc"])/3

x = df.index.values.astype(float)
y_values = np.array([val.nominal_value for val in df['avg']])
y_errors = np.array([val.std_dev for val in df['avg']])

def linear(x, m, c):
    return m * x + c

popt, pcov = curve_fit(linear, x, y_values, sigma=y_errors, absolute_sigma=True)

# 3. Re-pack into ufloat for your results
m = ufloat(popt[0], np.sqrt(pcov[0,0]))
c = ufloat(popt[1], np.sqrt(pcov[1,1]))

In [145]:
plt.rcParams["text.usetex"] = True

fig, ax = plt.subplots(figsize=(10, 6))
ax.xaxis.set_major_locator(MultipleLocator(1))

for col in df.columns:
    if col != "avg":
        ax.scatter(df.index, [x.nominal_value for x in df[col]], label="Messpunkte (" + col.split("_")[0] + " cm)")

# 2. Plot the regression line with uncertainty shading
x_fit = np.linspace(df.index.min(), df.index.max(), 100)
y_fit = m.nominal_value * x_fit + c.nominal_value

# Calculate the uncertainty of the fit line (simple linear error propagation)
y_fit_err = np.sqrt((x_fit * m.std_dev)**2 + c.std_dev**2)

ax.plot(x_fit, y_fit, "-", color="red", label=f"Fit: $m={m:.2uP}$, $c={c:.2uP}$")

ax.legend()
ax.set_xlabel("Ordnung")
ax.set_ylabel("Quotient s/l")
ax.grid(True)
plt.show()


RuntimeError: latex was not able to process the following string:
b'lp'

Here is the full command invocation and its output:

latex -interaction=nonstopmode --halt-on-error --output-directory=tmpo1l52d16 844f707e0917555faafa721cd5105dbd.tex

This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) (preloaded format=latex)
 restricted \write18 enabled.
entering extended mode
(./844f707e0917555faafa721cd5105dbd.tex
LaTeX2e <2024-11-01> patch level 2
L3 programming layer <2025-01-18>
(/usr/local/texlive/2025basic/texmf-dist/tex/latex/base/article.cls
Document Class: article 2024/06/29 v1.4n Standard LaTeX document class
(/usr/local/texlive/2025basic/texmf-dist/tex/latex/base/size10.clo))

! LaTeX Error: File `type1cm.sty' not found.

Type X to quit or <RETURN> to proceed,
or enter new name. (Default extension: sty)

Enter file name: 
! Emergency stop.
<read *> 
         
l.7 \usepackage
               {type1ec}^^M
No pages of output.
Transcript written on tmpo1l52d16/844f707e091755

<Figure size 1000x600 with 1 Axes>

In [123]:
print("Results:")
print("m: ", m)
print("c: ", c)

Results:
m:  0.00424+/-0.00009
c:  -0.00039+/-0.00028


In [139]:
lam = ufloat(532.1, 1) * 1e-9
d = lam/m
print(f"{d:.2uP}")

0.0001256±0.0000026
